In [ ]:
!pip install -q kagglehub pandas
!pip install spacy
!pip install -U sentence-transformers

import kagglehub
import pandas as pd
import os
import ast
import spacy
import numpy as np
import json
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.3.0
    Uninstalling sentence-transformers-5.3.0:
      Successfully uninstalled sentence-transformers-5.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load in News Dataset

In [ ]:
#Download MINDS dataset
path = kagglehub.dataset_download("arashnic/mind-news-dataset")

#Training Folder
train_path = os.path.join(path, "MINDsmall_train")
train_entity_path = os.path.join(train_path, "entity_embedding.vec")

#Load dataset
news = pd.read_csv(os.path.join(train_path, "news.tsv"), sep="\t", header=None)

#Add in the column names
news.columns = ["news_id","category","subcategory","title","abstract","url","title_entities","abstract_entities"]

#Total number of instances for news.tsv
print("Total instances in news:", len(news))

Using Colab cache for faster access to the 'mind-news-dataset' dataset.
Total instances in news: 51282


In [ ]:
def load_entity_embeddings(filepath):
    entity_lookup = {}

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue

            entity_id = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            entity_lookup[entity_id] = vector

    emb_dim = len(next(iter(entity_lookup.values())))
    return entity_lookup, emb_dim

entity_lookup, kg_emb_dim = load_entity_embeddings(train_entity_path)

#1. Two-Branch Entity Embedding

To enhance the semantic representation of news articles, a two-branch entity embedding framework is adopted. This approach captures entity-level information from two complementary sources:

*  Knowledge Graph (KG) branch – uses structured, dataset-provided entities from `entity_embedding.vec`
*  Named Entity Recognition (NER) branch – extracts entities directly from raw text

Each branch produces its own embedding, allowing the model to leverage both structured knowledge and text-derived signals.

Due to the presence flags, title will not be used as a substitute for any missing abstract values as it duplicates features and may introduce bias.

In [ ]:
# Converts all values of title_entities and abstract_entities into Python lists
def parse_entities(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            parsed = json.loads(x)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

##1.1 Handling Missing `title_entities` and `abstract_entities`

There are 13,846 instances of 'title_entities' and 13,829 instances of 'abstract_entities' missing.

For missing values in title_entities, named entities were extracted directly from the title text using a Named Entity Recognition (NER) model (spaCy). This ensures that entity information remains consistent and semantically meaningful, rather than introducing artificial or placeholder values. The column 'title_entities' which contains entity IDs for lookup will be renamed as 'kg_title_entities'. The NER extracted title entities are then stored in a new column 'ner_title_entities'.

Similarly, missing values in abstract_entities were filled by extracting entities from the abstract text using the same NER approach. The column 'abstract_entities' which contains entity IDs for lookup will be renamed as 'kg_abstract_entities'. The NER extracted abstract entities were then stored in a new column 'ner_abstract_entities'.

In [ ]:
# Extracts the WikidataID found in non-missing rows of title_entities and abstract_entities
def extract_entity_ids(entity_list, id_key="WikidataId"):
    if not entity_list:
        return []

    ids = []
    seen = set()

    for ent in entity_list:
        if not isinstance(ent, dict):
            continue

        entity_id = ent.get(id_key)
        if entity_id is None:
            continue

        entity_id = str(entity_id).strip()
        if not entity_id:
            continue

        if entity_id in seen:
            continue

        seen.add(entity_id)
        ids.append(entity_id)

    return ids

# Extracts entities from title and abstract text for missing rows using NER
# Load spaCy NER model
nlp = spacy.load("en_core_web_sm")

# Load sentence embedding model
text_emb_model = SentenceTransformer("all-MiniLM-L6-v2")
ner_emb_dim = text_emb_model.get_sentence_embedding_dimension()

def extract_entity_texts(text):
    if text is None:
        return []
    if isinstance(text, float) and pd.isna(text):
        return []

    text = str(text).strip()
    if text == "":
        return []

    doc = nlp(text)
    return [ent.text for ent in doc.ents]

def clean_entity_phrases(phrases):
    if not phrases:
        return []

    cleaned = []
    seen = set()

    for p in phrases:
        p = str(p).strip()
        if not p:
            continue

        key = p.lower()
        if key in seen:
            continue

        seen.add(key)
        cleaned.append(p)

    return cleaned

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_5640/3005351920.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  ner_emb_dim = text_emb_model.get_sentence_embedding_dimension()


After applying NER-based extraction, 3,717 for title entities and 4,324 for abstract entities still contain empty entity lists as not all news texts contain recognizable named entities. These cases were retained as empty lists as forcing artificial entities could introduce noise.

Subsequently, to generate entity embeddings for title and abstract entities, we attempted to augment entity coverage using NER and Wikidata-based entity linking. However, due to noisy entity extraction and unreliable linking quality, this approach did not produce meaningful improvements. As such, the final model relies on dataset-provided and text embedding of extracted title and abstract entities.

Thus, the Two-Branch Entity Embedding approach is used.

In [ ]:
# Branches are normalized to ensure that both branches contribute fairly
def normalize(vec):
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm

In [ ]:
# Extracts entity embeddings using Wikidata IDs
def entity_ids_to_embedding(entity_ids, entity_lookup, emb_dim):
    if not entity_ids:
        return np.zeros(emb_dim, dtype=np.float32)

    vecs = []
    for entity_id in entity_ids:
        vec = entity_lookup.get(entity_id)
        if vec is not None:
            vecs.append(vec)

    if not vecs:
        return np.zeros(emb_dim, dtype=np.float32)

    pooled = np.mean(np.stack(vecs, axis=0), axis=0).astype(np.float32)
    return normalize(pooled)

# Converts NER entities to embeddings
def phrases_to_embedding(phrases, model, emb_dim):
    if not phrases:
        return np.zeros(emb_dim, dtype=np.float32)

    vecs = model.encode(
        phrases,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    pooled = np.mean(vecs, axis=0).astype(np.float32)
    return normalize(pooled)

Due to the presence of missing entity annotations (approximately 27% of articles), zero-padding is used for absent embeddings. However, zero vectors are ambiguous and may be misinterpreted by the model as meaningful low-valued signals. To resolve this, presence flags are introduced to explicitly encode whether each branch (KG or NER) contains valid information. This enables the model to distinguish between missing and weak signals, improving robustness and learning efficiency.

In [ ]:
# Presence flag for mapped entities using Wikidata IDs
def has_mapped_entity(entity_ids, entity_lookup):
    return int(any(entity_id in entity_lookup for entity_id in entity_ids))

# Presence flag for extracted entities
def has_any_phrase(phrases):
    return 1 if phrases else 0

In [ ]:
# Parse raw entity columns
news["kg_title_entities"] = news["title_entities"].apply(parse_entities)
news["kg_abstract_entities"] = news["abstract_entities"].apply(parse_entities)

# Extract unique entity IDs
news["kg_title_ids"] = news["kg_title_entities"].apply(lambda x: extract_entity_ids(x, id_key="WikidataId"))
news["kg_abstract_ids"] = news["kg_abstract_entities"].apply(lambda x: extract_entity_ids(x, id_key="WikidataId"))

# Extract raw NER entities
news["ner_title_entities"] = news["title"].apply(extract_entity_texts)
news["ner_abstract_entities"] = news["abstract"].apply(extract_entity_texts)

# Clean phrases
news["ner_title_phrases"] = news["ner_title_entities"].apply(clean_entity_phrases)
news["ner_abstract_phrases"] = news["ner_abstract_entities"].apply(clean_entity_phrases)

# Presence flags for mapped entities
news["kg_title_flag"] = news["kg_title_ids"].apply(lambda x: has_mapped_entity(x, entity_lookup))
news["kg_abstract_flag"] = news["kg_abstract_ids"].apply(lambda x: has_mapped_entity(x, entity_lookup))

# Presence flags for extracted entities
news["ner_title_flag"] = news["ner_title_phrases"].apply(has_any_phrase)
news["ner_abstract_flag"] = news["ner_abstract_phrases"].apply(has_any_phrase)

# Pooled KG embeddings
news["kg_title_emb"] = news["kg_title_ids"].apply(
    lambda x: entity_ids_to_embedding(x, entity_lookup, kg_emb_dim)
)
news["kg_abstract_emb"] = news["kg_abstract_ids"].apply(
    lambda x: entity_ids_to_embedding(x, entity_lookup, kg_emb_dim)
)

# Pooled NER embeddings
news["ner_title_emb"] = news["ner_title_phrases"].apply(
    lambda x: phrases_to_embedding(x, text_emb_model, ner_emb_dim)
)
news["ner_abstract_emb"] = news["ner_abstract_phrases"].apply(
    lambda x: phrases_to_embedding(x, text_emb_model, ner_emb_dim)
)

#2. Extract File

In [ ]:
news_entities = news[[
    "news_id",

    # KG branch
    "kg_title_emb",
    "kg_abstract_emb",
    "kg_title_flag",
    "kg_abstract_flag",

    # NER branch
    "ner_title_emb",
    "ner_abstract_emb",
    "ner_title_flag",
    "ner_abstract_flag"
]].copy()

news_entities.to_csv(
    "/content/drive/MyDrive/embeddings/news_entities.csv",
    index=False
)
